# Marketing Campaign Analysis and Investment Recommendation

## Objective
Analyze `Marketing_Campaign_Data.csv` and identify the best campaign-tone and channel combination to invest in.

- Campaign A uses a casual tone.
- Campaign B uses a sales-oriented tone.
- The test covers one week.
- Every campaign-channel combination received the same investment.

Under that equal-spend assumption, the primary business metric is realized total sales. Conversion rate and sales per interaction are used as secondary checks.

## Dependencies

Run the next cell once to install the Python packages required by this notebook and the linked export scripts for the dashboard, presentation, Word report, and executive memo.

Current notebook dependency stack:

- `pandas` for dataset transformation and summary tables
- `numpy` for numerical support in analysis helpers
- `plotly` for notebook visuals and dashboard charts
- `scipy` for statistical support used in campaign testing
- `python-pptx` for stakeholder presentation generation
- `python-docx` for Word report and executive memo generation
- `ipython` for notebook display helpers

In [17]:
import subprocess
import sys

# Versions aligned with the current analysis and export scripts.
packages = [
    "pandas==3.0.2",
    "numpy==2.4.4",
    "plotly==6.7.0",
    "scipy==1.17.1",
    "python-pptx==1.0.2",
    "python-docx==1.2.0",
    "ipython",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    *packages,
])

print("Notebook analysis and export dependencies are installed.")

Notebook analysis and export dependencies are installed.


In [18]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

BASE_DIR = Path(r"c:\\Users\\davie\\OneDrive\\Documentos\\Data Analyst")
DATA_PATH = BASE_DIR / "Marketing_Campaign_Data.csv"
CONVERSION_COLUMN = "Converted (1=yes, 0=no)"
TIME_COLUMN = "Time on Site (seconds)"
SALES_COLUMN = "Sales ($)"
ID_COLUMN = "Interaction ID"
CAMPAIGN_LABELS = {"A": "Casual tone", "B": "Sales-oriented tone"}

df = pd.read_csv(DATA_PATH)
df.columns = [column.strip() for column in df.columns]
for column in [CONVERSION_COLUMN, TIME_COLUMN, SALES_COLUMN]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df[CONVERSION_COLUMN] = df[CONVERSION_COLUMN].astype(int)
df.head()

,Interaction ID,Campaign Type,Channel,Customer Type,"Converted (1=yes, 0=no)",Time on Site (seconds),Sales ($)
0,INT000001,B,Instagram,Existing,1,220.627803,44.796380
1,INT000002,B,Email,New,1,168.161435,61.085973
2,INT000003,A,Email,New,0,146.636771,0.000000
3,INT000004,B,Email,Existing,1,106.726457,37.556561
4,INT000005,B,Web Banner,Existing,1,151.121076,30.769231


In [19]:
quality_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "missing_values",
        "duplicate_interaction_ids",
        "overall_conversion_rate",
        "average_sales_per_interaction",
        "average_time_on_site_seconds",
    ],
    "value": [
        len(df),
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df[ID_COLUMN].duplicated().sum()),
        round(df[CONVERSION_COLUMN].mean(), 4),
        round(df[SALES_COLUMN].mean(), 2),
        round(df[TIME_COLUMN].mean(), 2),
    ]
})

display(Markdown("## Dataset Health"))
display(quality_summary)
display(Markdown("### Column Types"))
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))

## Dataset Health

,metric,value
0,rows,100000.0000
1,columns,7.0000
2,missing_values,0.0000
3,duplicate_interaction_ids,0.0000
4,overall_conversion_rate,0.4771
5,average_sales_per_interaction,23.1500
6,average_time_on_site_seconds,134.5700


### Column Types

,column,dtype
0,Interaction ID,str
1,Campaign Type,str
2,Channel,str
3,Customer Type,str
4,"Converted (1=yes, 0=no)",int64
5,Time on Site (seconds),float64
6,Sales ($),float64


In [20]:
def normal_cdf(value):
    return 0.5 * (1.0 + math.erf(value / math.sqrt(2.0)))


def two_proportion_test(successes_a, trials_a, successes_b, trials_b):
    pooled_rate = (successes_a + successes_b) / (trials_a + trials_b)
    standard_error = math.sqrt(pooled_rate * (1.0 - pooled_rate) * ((1.0 / trials_a) + (1.0 / trials_b)))
    if standard_error == 0:
        return 0.0, 1.0
    z_score = ((successes_a / trials_a) - (successes_b / trials_b)) / standard_error
    p_value = 2.0 * (1.0 - normal_cdf(abs(z_score)))
    return z_score, p_value


def summarize(data_frame, group_columns):
    return (
        data_frame.groupby(group_columns, as_index=False)
        .agg(
            leads=(ID_COLUMN, "count"),
            conversions=(CONVERSION_COLUMN, "sum"),
            conversion_rate=(CONVERSION_COLUMN, "mean"),
            total_sales=(SALES_COLUMN, "sum"),
            sales_per_lead=(SALES_COLUMN, "mean"),
            avg_time_on_site=(TIME_COLUMN, "mean"),
        )
        .sort_values(["total_sales", "sales_per_lead", "conversion_rate"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

In [21]:
customer_summary = summarize(df, ["Customer Type"])
overall_campaign_summary = summarize(df, ["Campaign Type"])
overall_channel_summary = summarize(df, ["Channel"])
overall_combo_summary = summarize(df, ["Campaign Type", "Channel"])
new_customers = df[df["Customer Type"].eq("New")].copy()
new_combo_summary = summarize(new_customers, ["Campaign Type", "Channel"])

overall_combo_summary["revenue_share"] = overall_combo_summary["total_sales"] / overall_combo_summary["total_sales"].sum()

display(Markdown("## EDA Summary Tables"))
display(Markdown("### Customer Type Summary"))
display(customer_summary.round(4))
display(Markdown("### Campaign Summary"))
display(overall_campaign_summary.round(4))
display(Markdown("### Channel Summary"))
display(overall_channel_summary.round(4))
display(Markdown("### Campaign and Channel Ranking"))
display(overall_combo_summary.round(4))

## EDA Summary Tables

### Customer Type Summary

,Customer Type,leads,conversions,conversion_rate,total_sales,sales_per_lead,avg_time_on_site
0,Existing,57403,27319,0.4759,1.326214e+06,23.1036,134.4193
1,New,42597,20389,0.4786,9.885285e+05,23.2065,134.7648


### Campaign Summary

,Campaign Type,leads,conversions,conversion_rate,total_sales,sales_per_lead,avg_time_on_site
0,B,56218,26817,0.4770,1.300870e+06,23.1397,134.7527
1,A,43782,20891,0.4772,1.013872e+06,23.1573,134.3274


### Channel Summary

,Channel,leads,conversions,conversion_rate,total_sales,sales_per_lead,avg_time_on_site
0,Email,41136,19532,0.4748,947748.8688,23.0394,134.4828
1,Instagram,29339,14130,0.4816,685223.0769,23.3554,134.8955
2,Web Banner,29525,14046,0.4757,681770.5882,23.0913,134.3562


### Campaign and Channel Ranking

,Campaign Type,Channel,leads,conversions,conversion_rate,total_sales,sales_per_lead,avg_time_on_site,revenue_share
0,B,Email,23065,11019,0.4777,533036.6516,23.1102,134.4376,0.2303
1,A,Email,18071,8513,0.4711,414712.2172,22.9490,134.5404,0.1792
2,B,Web Banner,16694,7938,0.4755,385345.7014,23.0829,134.8608,0.1665
3,B,Instagram,16459,7860,0.4776,382487.7828,23.2388,135.0844,0.1652
4,A,Instagram,12880,6270,0.4868,302735.2941,23.5043,134.6541,0.1308
5,A,Web Banner,12831,6108,0.4760,296424.8869,23.1022,133.6997,0.1281


In [22]:
chart_data = overall_combo_summary.copy()
chart_data["combination"] = chart_data["Campaign Type"] + " + " + chart_data["Channel"]
chart_data["tone"] = chart_data["Campaign Type"].map(CAMPAIGN_LABELS)

fig_sales = px.bar(
    chart_data,
    x="Channel",
    y="total_sales",
    color="Campaign Type",
    barmode="group",
    text_auto=".2s",
    title="Overall Revenue by Campaign and Channel",
    color_discrete_map={"A": "#c96b59", "B": "#2f7f7b"},
)
fig_sales.update_layout(template="plotly_white", yaxis_title="Sales ($)")
fig_sales.show()

fig_efficiency = px.scatter(
    chart_data,
    x="conversion_rate",
    y="sales_per_lead",
    size="leads",
    color="Campaign Type",
    hover_name="combination",
    text="combination",
    title="Scale vs Efficiency by Combination",
    color_discrete_map={"A": "#c96b59", "B": "#2f7f7b"},
    size_max=60,
)
fig_efficiency.update_traces(textposition="top center")
fig_efficiency.update_layout(template="plotly_white", xaxis_tickformat=".1%", yaxis_title="Sales per Interaction ($)")
fig_efficiency.show()

customer_mix = df.groupby("Customer Type", as_index=False).agg(
    interactions=(ID_COLUMN, "count"),
    total_sales=(SALES_COLUMN, "sum"),
)
fig_customer = go.Figure()
fig_customer.add_bar(x=customer_mix["Customer Type"], y=customer_mix["total_sales"], name="Sales", marker_color="#20364f")
fig_customer.add_scatter(x=customer_mix["Customer Type"], y=customer_mix["interactions"], name="Interactions", mode="lines+markers", yaxis="y2", marker_color="#c96b59")
fig_customer.update_layout(
    template="plotly_white",
    title="Customer Mix: Revenue and Interaction Volume",
    yaxis=dict(title="Sales ($)"),
    yaxis2=dict(title="Interactions", overlaying="y", side="right"),
)
fig_customer.show()

In [23]:
display(Markdown("## New Customer Analysis"))
display(new_combo_summary.round(4))

new_chart = new_combo_summary.copy()
new_chart["combination"] = new_chart["Campaign Type"] + " + " + new_chart["Channel"]

fig_new = px.bar(
    new_chart,
    x="Channel",
    y="total_sales",
    color="Campaign Type",
    barmode="group",
    text_auto=".2s",
    title="New-Customer Revenue by Campaign and Channel",
    color_discrete_map={"A": "#c96b59", "B": "#2f7f7b"},
)
fig_new.update_layout(template="plotly_white", yaxis_title="Sales ($)")
fig_new.show()

display(Markdown("Email remains the top revenue channel for new-customer acquisition, while Instagram remains the most efficient channel on a sales-per-interaction basis."))

## New Customer Analysis

,Campaign Type,Channel,leads,conversions,conversion_rate,total_sales,sales_per_lead,avg_time_on_site
0,B,Email,9700,4673,0.4818,225285.5204,23.2253,134.5451
1,A,Email,7683,3628,0.4722,177366.9683,23.0856,134.6997
2,B,Instagram,7100,3429,0.4830,166722.1719,23.4820,134.7529
3,B,Web Banner,7110,3387,0.4764,164125.7919,23.0838,135.7395
4,A,Instagram,5456,2658,0.4872,127814.9321,23.4265,134.9131
5,A,Web Banner,5548,2614,0.4712,127213.1222,22.9295,133.8595


Email remains the top revenue channel for new-customer acquisition, while Instagram remains the most efficient channel on a sales-per-interaction basis.

In [24]:
overall_campaign_lookup = overall_campaign_summary.set_index("Campaign Type")
new_campaign_summary = summarize(new_customers, ["Campaign Type"])
new_campaign_lookup = new_campaign_summary.set_index("Campaign Type")

overall_z, overall_p = two_proportion_test(
    int(overall_campaign_lookup.loc["B", "conversions"]),
    int(overall_campaign_lookup.loc["B", "leads"]),
    int(overall_campaign_lookup.loc["A", "conversions"]),
    int(overall_campaign_lookup.loc["A", "leads"]),
)
new_z, new_p = two_proportion_test(
    int(new_campaign_lookup.loc["B", "conversions"]),
    int(new_campaign_lookup.loc["B", "leads"]),
    int(new_campaign_lookup.loc["A", "conversions"]),
    int(new_campaign_lookup.loc["A", "leads"]),
)

pivot = overall_combo_summary.pivot(index="Channel", columns="Campaign Type")
tone_deltas = []
for channel in pivot.index:
    z_score, p_value = two_proportion_test(
        int(pivot.loc[channel, ("conversions", "B")]),
        int(pivot.loc[channel, ("leads", "B")]),
        int(pivot.loc[channel, ("conversions", "A")]),
        int(pivot.loc[channel, ("leads", "A")]),
    )
    tone_deltas.append({
        "Channel": channel,
        "B_minus_A_sales": float(pivot.loc[channel, ("total_sales", "B")] - pivot.loc[channel, ("total_sales", "A")]),
        "B_minus_A_conversion_pp": float((pivot.loc[channel, ("conversion_rate", "B")] - pivot.loc[channel, ("conversion_rate", "A")]) * 100),
        "B_minus_A_sales_per_lead": float(pivot.loc[channel, ("sales_per_lead", "B")] - pivot.loc[channel, ("sales_per_lead", "A")]),
        "p_value": float(p_value),
    })
tone_deltas = pd.DataFrame(tone_deltas).sort_values("B_minus_A_sales", ascending=False).reset_index(drop=True)

display(Markdown("## Statistical Check"))
display(pd.DataFrame({
    "test": ["Campaign B vs A overall conversion", "Campaign B vs A new-customer conversion"],
    "z_score": [round(overall_z, 3), round(new_z, 3)],
    "p_value": [round(overall_p, 4), round(new_p, 4)],
}))
display(Markdown("The conversion-rate gap between Campaign B and Campaign A is not statistically decisive at campaign level, so the investment recommendation is driven mainly by realized revenue and scale, not by a dramatic conversion-rate lift."))
display(tone_deltas.round(4))

## Statistical Check

,test,z_score,p_value
0,Campaign B vs A overall conversion,-0.044,0.9645
1,Campaign B vs A new-customer conversion,0.870,0.3843


The conversion-rate gap between Campaign B and Campaign A is not statistically decisive at campaign level, so the investment recommendation is driven mainly by realized revenue and scale, not by a dramatic conversion-rate lift.

,Channel,B_minus_A_sales,B_minus_A_conversion_pp,B_minus_A_sales_per_lead,p_value
0,Email,118324.4344,0.6651,0.1611,0.1801
1,Web Banner,88920.8145,-0.0534,-0.0194,0.9274
2,Instagram,79752.4887,-0.9251,-0.2655,0.1155


In [25]:
top_overall = overall_combo_summary.iloc[0]
runner_up = overall_combo_summary.iloc[1]
top_new = new_combo_summary.iloc[0]
best_efficiency = overall_combo_summary.sort_values("sales_per_lead", ascending=False).iloc[0]

decision_table = pd.DataFrame([
    {
        "priority": "Primary investment",
        "combination": f"{top_overall['Campaign Type']} + {top_overall['Channel']}",
        "reason": "Highest total weekly sales under equal spend",
    },
    {
        "priority": "Secondary optimization lane",
        "combination": "B + Instagram",
        "reason": "Strong sales with high efficiency and social reach",
    },
    {
        "priority": "Benchmark control",
        "combination": "A + Instagram",
        "reason": "Best-performing casual-tone benchmark",
    },
])

display(Markdown("## Final Recommendation"))
display(decision_table)
display(Markdown(
    f"**Invest in Campaign B + Email.** It generated ${top_overall['total_sales']:,.2f} in one week from {int(top_overall['leads']):,} interactions, with a {top_overall['conversion_rate']:.2%} conversion rate. "
    f"It also ranked first among new customers with ${top_new['total_sales']:,.2f} in sales. The runner-up was {runner_up['Campaign Type']} + {runner_up['Channel']}, but it trailed by ${top_overall['total_sales'] - runner_up['total_sales']:,.2f}. "
    f"For a secondary test lane, keep B + Instagram because it combines strong scale with better per-interaction efficiency than the top option."
))

## Final Recommendation

,priority,combination,reason
0,Primary investment,B + Email,Highest total weekly sales under equal spend
1,Secondary optimization lane,B + Instagram,Strong sales with high efficiency and social r...
2,Benchmark control,A + Instagram,Best-performing casual-tone benchmark


**Invest in Campaign B + Email.** It generated $533,036.65 in one week from 23,065 interactions, with a 47.77% conversion rate. It also ranked first among new customers with $225,285.52 in sales. The runner-up was A + Email, but it trailed by $118,324.43. For a secondary test lane, keep B + Instagram because it combines strong scale with better per-interaction efficiency than the top option.

In [26]:
from generate_marketing_recommendation_presentation import analyze, build_presentation, write_dashboard, write_markdown_report

metrics = analyze(df)
write_markdown_report(metrics)
write_dashboard(metrics)
presentation = build_presentation(metrics)
presentation_path = BASE_DIR / "Marketing_Campaign_Stakeholder_Presentation.pptx"
presentation.save(presentation_path)

deliverables = pd.DataFrame({
    "deliverable": ["analysis report", "dashboard", "dashboard copy", "presentation"],
    "path": [
        str(BASE_DIR / "marketing_campaign_analysis.md"),
        str(BASE_DIR / "dashboard.html"),
        str(BASE_DIR / "marketing_campaign_dashboard.html"),
        str(presentation_path),
    ]
})
display(Markdown("## Exported Deliverables"))
display(deliverables)

## Exported Deliverables

,deliverable,path
0,analysis report,c:\Users\davie\OneDrive\Documentos\Data Analys...
1,dashboard,c:\Users\davie\OneDrive\Documentos\Data Analys...
2,dashboard copy,c:\Users\davie\OneDrive\Documentos\Data Analys...
3,presentation,c:\Users\davie\OneDrive\Documentos\Data Analys...


## Stakeholder Summary

- Best primary investment: **Campaign B + Email**
- Why: highest weekly revenue under equal spend, same winner in the new-customer slice
- Secondary lane: **Campaign B + Instagram** for efficiency-oriented follow-up testing
- Caution: this is a revenue-based recommendation because the dataset does not include spend, CPA, or profit margin

In [27]:
import importlib

import pandas as pd

from IPython.display import Markdown, display

import generate_marketing_stakeholder_materials as stakeholder_materials
import generate_marketing_word_report as marketing_word_report

stakeholder_materials = importlib.reload(stakeholder_materials)
marketing_word_report = importlib.reload(marketing_word_report)

word_report_path = marketing_word_report.build_report()
executive_memo_path = marketing_word_report.build_executive_memo()
stakeholder_materials.main()

word_deliverable = pd.DataFrame({
    "deliverable": [
        "word report",
        "executive memo",
        "meeting brief (markdown)",
        "stakeholder Q&A (markdown)",
        "presenter script (markdown)",
        "meeting brief (word)",
        "stakeholder Q&A (word)",
    ],
    "path": [
        str(word_report_path),
        str(executive_memo_path),
        str(stakeholder_materials.MEETING_BRIEF_PATH),
        str(stakeholder_materials.QA_PATH),
        str(stakeholder_materials.SCRIPT_PATH),
        str(stakeholder_materials.MEETING_BRIEF_DOCX_PATH),
        str(stakeholder_materials.QA_DOCX_PATH),
    ],
})

display(Markdown("## Stakeholder Deliverables Export"))
display(word_deliverable)

C:\Users\davie\OneDrive\Documentos\Data Analyst\Marketing_Campaign_Stakeholder_Meeting_Brief.md
C:\Users\davie\OneDrive\Documentos\Data Analyst\Marketing_Campaign_Stakeholder_QA.md
C:\Users\davie\OneDrive\Documentos\Data Analyst\Marketing_Campaign_Presenter_Script.md
C:\Users\davie\OneDrive\Documentos\Data Analyst\Marketing_Campaign_Stakeholder_Meeting_Brief.docx
C:\Users\davie\OneDrive\Documentos\Data Analyst\Marketing_Campaign_Stakeholder_QA.docx


## Stakeholder Deliverables Export

,deliverable,path
0,word report,C:\Users\davie\OneDrive\Documentos\Data Analys...
1,executive memo,C:\Users\davie\OneDrive\Documentos\Data Analys...
2,meeting brief (markdown),C:\Users\davie\OneDrive\Documentos\Data Analys...
3,stakeholder Q&A (markdown),C:\Users\davie\OneDrive\Documentos\Data Analys...
4,presenter script (markdown),C:\Users\davie\OneDrive\Documentos\Data Analys...
5,meeting brief (word),C:\Users\davie\OneDrive\Documentos\Data Analys...
6,stakeholder Q&A (word),C:\Users\davie\OneDrive\Documentos\Data Analys...
